# Задание: Сжатие изображений с помощью SVD (Singular Value Decomposition)

## 1. Цель работы
Научиться применять линейную алгебру (SVD) для решения прикладной задачи — сжатия растровых изображений с контролируемой потерей качества. Вы должны не просто запустить код, а проанализировать, **как ранг аппроксимации влияет на качество и размер файла**.

## 2. Теоретическая справка (кратко)
Любую матрицу $ A $ размером $ m \times n $ можно разложить на три матрицы:
$
A = U \cdot \Sigma \cdot V^T
$
Где:
- $ U $ — ортогональная матрица размера $ m \times m $ (левые сингулярные векторы).
- $ \Sigma $ — диагональная матрица размера $ m \times n $, на диагонали которой стоят сингулярные числа $ \sigma_1 \ge \sigma_2 \ge ... \ge \sigma_k $.
- $ V^T $ — ортогональная матрица размера $ n \times n $ (правые сингулярные векторы).

**Идея сжатия:** Если оставить только первые $ r $ сингулярных чисел (и соответствующие столбцы U и V), мы получим матрицу $ A_r $, которая является наилучшим приближением $ A $ среди всех матриц ранга $ r $ (в смысле нормы Фробениуса).

## 3. Техническое задание (Пошаговый план)

### Часть А. Загрузка и подготовка данных (Базовый уровень)
1. **Выбор изображения:**
   - Загрузите любую картинку (можно стандартный `lena` или `peppers`, либо свою фотографию).
   - *Усложнение:* Картинка должна быть цветной, размером не менее 300x300 пикселей.
2. **Чтение файла:**
   - Используйте `PIL.Image.open()` или `matplotlib.image.imread()`.
3. **Преобразование в массив:**
   - Переведите картинку в формат `numpy.ndarray`.
4. **Работа с цветом (выбор стратегии):**
   - *Вариант А (простой):* Переведите изображение в градации серого (одноканальное) с помощью `.convert('L')` или формулы `Gray = 0.299*R + 0.587*G + 0.114*B`. Тогда у вас будет одна матрица $ A $ размером $ height \times width $.
   - *Вариант Б (продвинутый):* Оставьте 3 цветовых канала (RGB). Примените SVD отдельно к матрице **каждого** канала (Red, Green, Blue). Затем соберите их обратно.

### Часть Б. Применение SVD и восстановление
1. **Разложение:**
   - Используйте функцию `np.linalg.svd(матрица, full_matrices=False)`.
   - *Внимание:* Параметр `full_matrices=False` важен для экономии памяти, иначе матрица $ U $ получится слишком большой.
2. **Сжатие (аппроксимация) для разных рангов:**
   - Восстановите изображение, используя только `r` сингулярных чисел.
   - Обязательные значения `r`: **[1, 5, 10, 30, 100, min(shape)]** (где последнее — полное восстановление без потерь).
   - Формула восстановления:
     $
     A_{approx} = U[:, :r] \cdot diag(\Sigma[:r]) \cdot V[:r, :]^T
     $
   - *Подсказка:* `np.diag(s[:r])` создаст диагональную матрицу, но для ускорения лучше использовать `(U[:, :r] * s[:r]) @ V[:r, :]`.

### Часть В. Визуализация результатов
- Создайте один большой график (subplot) размером 2x3 или 3x2.
- На каждом подграфике отобразите восстановленное изображение.
- **Подпишите** каждый подграфик строкой: `"Ранг = r, Сжатие в X раз"`.
- Оригинальное изображение выведите отдельно (или как самый последний график).

### Часть Г. Подсчет эффективности сжатия (Анализ размера)
Здесь важно понять, сколько **байт** нужно сохранить для хранения сжатой версии.
- Исходное изображение (RGB) весит: $ 3 \times height \times width $ байт (если считать 1 байт на канал).
- Сжатая версия хранится в виде трех матриц: $ U_r $, $ \Sigma_r $, $ V^T_r $.
- Размер хранения:
  - $ U_r $: $ height \times r $ чисел.
  - $ \Sigma_r $: $ r $ чисел (диагональ).
  - $ V^T_r $: $ r \times width $ чисел.
- **Итоговая формула для хранения (в числах с плавающей точкой):** $ r \times (height + 1 + width) $.
  - *Для цветной картинки:* умножьте это на 3 (по количеству каналов).
- **Сравнение:** Подсчитайте коэффициент сжатия: $ \frac{Исходный\_размер\_в\_байтах}{Размер\_сжатого\_представления\_в\_байтах} $.
  - *Важно:* Учтите, что в `numpy` числа float64 занимают 8 байт, а в PNG — 1 байт на канал. Сделайте допущение, что мы храним именно `float32` (4 байта) для сжатого представления.

---

## 4. Обязательная часть: Автоматическая проверка (Self-Check)

Чтобы вы (студент) могли убедиться, что задача решена верно, вставьте в конце ноутбука следующие проверки (asserts). **Преподаватель будет смотреть на прохождение этих проверок.**

```python
# ---------- БЛОК АВТОПРОВЕРКИ (НЕ РЕДАКТИРОВАТЬ) ----------
# Предполагается, что у вас есть переменные:
# original_shape = (h, w, c) или (h, w)
# list_of_ranks = [1, 5, 10, 30, 100, ...]
# approximations = список восстановленных массивов (numpy) для каждого ранга

# 1. Проверка размерностей
for i, r in enumerate(list_of_ranks):
    assert approximations[i].shape == original_shape, f"Ошибка: размерность для ранга {r} не совпадает с исходной"

# 2. Проверка, что значения не выходят за пределы 0-255 (для RGB/серого)
for i, r in enumerate(list_of_ranks):
    arr = approximations[i]
    if arr.dtype != np.uint8:
        arr = np.clip(arr, 0, 255) # Если float, то проверяем диапазон
    assert arr.max() <= 255.1, f"Значения превышают 255 для ранга {r}"
    assert arr.min() >= -0.1, f"Значения меньше 0 для ранга {r}"

# 3. Проверка, что с ростом ранга ошибка уменьшается (метрика MSE)
from sklearn.metrics import mean_squared_error
mse_list = []
for i in range(len(approximations)):
    # Если цветная, считаем MSE по всем каналам
    mse = mean_squared_error(original_flat, approximations[i].flatten())
    mse_list.append(mse)

# Проверяем монотонность (MSE должен падать, так как мы добавляем сингулярные числа)
for i in range(1, len(mse_list)):
    assert mse_list[i] <= mse_list[i-1] + 1e-6, f"MSE не уменьшается между рангами {list_of_ranks[i-1]} и {list_of_ranks[i]}"

print("✅ Все автоматические проверки пройдены. Задание выполнено корректно!")
```

---

## 5. Критерии оценки (Для преподавателя)

*Максимум: 10 баллов.*

1. **Корректность кода (3 балла)**:
   - Ноутбук запускается от начала до конца без ошибок.
   - Все блоки автопроверки (Assertions) проходят успешно.
2. **Качество визуализации (2 балла)**:
   - Выведены все 6 графиков с подписями.
   - Подписи содержат реальный коэффициент сжатия (не просто надпись "сжато").
3. **Глубина анализа (3 балла)**:
   - Студент написал **текстовый вывод** (Markdown ячейка) с ответами на вопросы:
     - При каком ранге визуальное качество перестает отличаться от оригинала?
     - Для какого ранга достигается сжатие в 10 раз?
     - Почему для темных/светлых участков ошибка разная?
4. **Продвинутая часть (бонус +2 балла)**:
   - Реализовано SVD для **RGB каналов отдельно**, а не перевод в серый.
   - Код работает быстро (использование broadcasting вместо циклов при сборке матриц).

---

## 6. Полезные функции и библиотеки (шпаргалка)

```python
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import mean_squared_error

# Чтение
img = Image.open('my_pic.jpg')
img_gray = img.convert('L')  # Серый
img_array = np.array(img_gray) # или np.array(img) для RGB

# SVD
U, s, Vt = np.linalg.svd(matrix, full_matrices=False)

# Восстановление для ранга r
U_r = U[:, :r]
S_r = np.diag(s[:r])
Vt_r = Vt[:r, :]
reconstructed = U_r @ S_r @ Vt_r

# Или быстрее (без создания диагональной матрицы):
reconstructed = (U[:, :r] * s[:r]) @ Vt[:r, :]

# Визуализация
plt.imshow(reconstructed, cmap='gray') # для серого
plt.imshow(reconstructed.astype(np.uint8)) # для RGB (привести к целым)
plt.axis('off')
plt.show()
```

---

### Совет по оформлению финального результата
В конце каждого блока задания 3 А-Г вставьте ячейку с типом **Markdown** и опишите свои результаты, а также добавьте в конце решения отдельную ячейку и назовите её **Выводы** и напиши общие выводы по работе и полученным результатам. Это должно приучить вас оформлять результаты исследований, а не просто сдавать «простыню кода». Такой текст уже не выглядит как «сделали что-то там», а превращается в полноценное инженерное исследование с четкими контрольными точками. Успехов!

In [ ]:
# Импорт библиотек и настройка отображения
import time
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image
from sklearn.metrics import mean_squared_error

plt.rcParams["figure.figsize"] = (15, 10)
plt.rcParams["font.size"] = 11

print("Библиотеки загружены")

## Часть А. Загрузка и подготовка изображения

Для работы я оставляю изображение цветным и применяю SVD отдельно к каждому из трёх RGB-каналов. Если интернет недоступен, код использует небольшое тестовое изображение с градиентами, поэтому ноутбук всё равно запускается от начала до конца.

In [ ]:
def make_fallback_image(size=512):
    """Создаёт воспроизводимое тестовое RGB-изображение без внешних файлов."""
    y, x = np.mgrid[0:size, 0:size]
    image = np.zeros((size, size, 3), dtype=np.float64)
    image[:, :, 0] = 255 * y / (size - 1)
    image[:, :, 1] = 255 * x / (size - 1)
    image[:, :, 2] = 255 * (x + y) / (2 * (size - 1))

    # Добавляем несколько контрастных областей, чтобы были заметны детали.
    image[70:190, 80:210] = [220, 70, 70]
    image[300:430, 300:450] = [50, 90, 220]
    image[210:330, 170:310] = 255 - image[210:330, 170:310]
    return Image.fromarray(np.clip(image, 0, 255).astype(np.uint8), mode="RGB")


image_url = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"

try:
    response = requests.get(image_url, timeout=10)
    response.raise_for_status()
    image = Image.open(BytesIO(response.content)).convert("RGB")
    print("Изображение загружено из интернета")
except Exception as error:
    print(f"Не удалось загрузить изображение ({type(error).__name__}). Используется тестовый вариант.")
    image = make_fallback_image()

# Для выполнения условия задания приводим изображение к размеру не меньше 300x300.
image = image.resize((512, 512))
img_rgb = np.array(image, dtype=np.uint8)
height, width, channels = img_rgb.shape

print(f"Размер изображения: {height} x {width} x {channels}")
print(f"Диапазон значений пикселей: [{img_rgb.min()}, {img_rgb.max()}]")

plt.figure(figsize=(7, 7))
plt.imshow(img_rgb)
plt.title("Исходное изображение")
plt.axis("off")
plt.show()

## Часть Б. SVD и восстановление изображения

Для каждого канала оставляю только первые `r` сингулярных чисел. Чем больше `r`, тем точнее восстановление и тем больше объём сжатого представления.

In [ ]:
def compress_channel(channel, rank):
    """Возвращает приближение одного канала рангом rank."""
    U, singular_values, Vt = np.linalg.svd(channel, full_matrices=False)
    rank = min(rank, len(singular_values))
    approximation = (U[:, :rank] * singular_values[:rank]) @ Vt[:rank, :]
    return np.clip(approximation, 0, 255)


def compress_rgb(rgb_array, rank):
    """Применяет SVD к каждому RGB-каналу и собирает изображение обратно."""
    compressed_channels = []
    for channel_index in range(rgb_array.shape[2]):
        channel = rgb_array[:, :, channel_index].astype(np.float64)
        compressed_channels.append(compress_channel(channel, rank))
    return np.stack(compressed_channels, axis=2)


def compression_ratio(image_shape, rank, bytes_per_number=4):
    """Коэффициент сжатия при хранении U, s и Vt в формате float32."""
    h, w, channel_count = image_shape
    original_size = h * w * channel_count  # uint8: 1 байт на канал
    compressed_size = rank * (h + 1 + w) * bytes_per_number * channel_count
    return original_size / compressed_size


# Обязательные ранги из задания, включая ранг 100 и полное восстановление.
ranks = list(dict.fromkeys([1, 5, 10, 30, 100, min(height, width)]))
approximations = []
compression_ratios = []
times = []

for rank in ranks:
    start_time = time.perf_counter()
    approximation = compress_rgb(img_rgb, rank)
    elapsed = time.perf_counter() - start_time

    approximations.append(approximation)
    compression_ratios.append(compression_ratio(img_rgb.shape, rank))
    times.append(elapsed)

print("Результаты вычислений:")
for rank, ratio, elapsed in zip(ranks, compression_ratios, times):
    print(f"  r = {rank:>3}: сжатие {ratio:>6.2f}x, время {elapsed:.3f} с")

In [ ]:
def as_uint8(array):
    """Подготавливает приближение к показу и сохранению как RGB uint8."""
    return np.clip(array, 0, 255).round().astype(np.uint8)


mse_list = [
    mean_squared_error(img_rgb.astype(np.float64).ravel(), approximation.ravel())
    for approximation in approximations
]

fig, axes = plt.subplots(2, 4, figsize=(16, 9))
axes = axes.ravel()

for axis, rank, approximation, ratio in zip(axes, ranks, approximations, compression_ratios):
    axis.imshow(as_uint8(approximation))
    axis.set_title(f"Ранг = {rank}\nСжатие в {ratio:.2f} раза")
    axis.axis("off")

axes[-1].imshow(img_rgb)
axes[-1].set_title("Оригинал")
axes[-1].axis("off")

plt.suptitle("Восстановление изображения при разных рангах SVD", fontsize=15)
plt.tight_layout()
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(ranks, mse_list, "o-", color="royalblue")
ax1.set_xscale("log")
ax1.set_xlabel("Ранг r")
ax1.set_ylabel("MSE")
ax1.set_title("Ошибка восстановления")
ax1.grid(alpha=0.3)

ax2.plot(ranks, compression_ratios, "o-", color="seagreen")
ax2.set_xscale("log")
ax2.set_xlabel("Ранг r")
ax2.set_ylabel("Коэффициент сжатия")
ax2.set_title("Коэффициент сжатия")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Часть Г. Анализ размера и качества

В расчёте считаю исходное RGB-изображение как набор `uint8` (1 байт на канал), а матрицы `U`, `s` и `Vt` — как `float32` (4 байта на число). Это теоретический размер хранения разложения, без учёта заголовков и служебной информации формата файла.

In [ ]:
original_size_bytes = img_rgb.nbytes
print(f"Размер исходного массива: {original_size_bytes:,} байт")
print(f"{'Ранг':>6} | {'Размер U,s,Vt':>16} | {'Сжатие':>10} | {'Экономия':>10} | {'MSE':>12}")
print("-" * 68)

for rank, ratio, mse in zip(ranks, compression_ratios, mse_list):
    compressed_size = int(rank * (height + 1 + width) * 4 * channels)
    savings = (1 - compressed_size / original_size_bytes) * 100
    print(f"{rank:>6} | {compressed_size:>16,} | {ratio:>9.2f}x | {savings:>9.1f}% | {mse:>12.2f}")

target_ratio = 10
target_indices = [i for i, ratio in enumerate(compression_ratios) if ratio >= target_ratio]
if target_indices:
    target_index = target_indices[-1]
    print(
        f"\nСжатие не менее чем в {target_ratio} раз сохраняется до "
        f"ранга r = {ranks[target_index]} (коэффициент {compression_ratios[target_index]:.2f}x)."
    )
else:
    print(f"\nДля выбранных рангов сжатие в {target_ratio} раз не достигнуто.")

## Обязательная автопроверка

Проверяю совпадение размерностей, диапазон значений пикселей и уменьшение MSE при увеличении ранга.

In [ ]:
# ---------- БЛОК АВТОПРОВЕРКИ (НЕ РЕДАКТИРОВАТЬ) ----------
original_shape = img_rgb.shape
list_of_ranks = ranks
approximations_list = approximations

for rank, approximation in zip(list_of_ranks, approximations_list):
    assert approximation.shape == original_shape, (
        f"Ошибка: размерность для ранга {rank} не совпадает с исходной"
    )
print("Проверка 1 пройдена: размерности совпадают")

for rank, approximation in zip(list_of_ranks, approximations_list):
    assert approximation.max() <= 255.1, f"Значения превышают 255 для ранга {rank}"
    assert approximation.min() >= -0.1, f"Значения меньше 0 для ранга {rank}"
print("Проверка 2 пройдена: значения находятся в диапазоне [0, 255]")

mse_list_check = [
    mean_squared_error(img_rgb.astype(np.float64).ravel(), approximation.ravel())
    for approximation in approximations_list
]
for previous, current, previous_rank, current_rank in zip(
    mse_list_check, mse_list_check[1:], list_of_ranks, list_of_ranks[1:]
):
    assert current <= previous + 1e-6, (
        f"MSE не уменьшается между рангами {previous_rank} и {current_rank}"
    )
print("Проверка 3 пройдена: MSE монотонно убывает")
print("\nВсе автоматические проверки пройдены. Решение работает корректно!")

## Выводы

SVD позволяет управлять компромиссом между качеством изображения и объёмом хранения. При малом ранге сохраняются главным образом крупные цветовые области и плавные изменения, но мелкие детали теряются. С увеличением ранга MSE уменьшается, а восстановленное изображение становится ближе к оригиналу.

Для выбранного изображения визуальное качество можно считать практически достаточным при ранге, после которого MSE уже меняется незначительно. Коэффициент сжатия в 10 раз определяется не «на глаз», а формулой из задания и зависит от размеров изображения и выбранного ранга.

SVD лучше сохраняет однородные области с низкой пространственной частотой. В областях с текстурами, резкими границами и мелкими деталями ошибка обычно выше, потому что такие особенности требуют большего количества сингулярных компонент. В отличие от JPEG, этот метод удобен для демонстрации математической идеи, но не учитывает особенности человеческого зрения и может быть менее эффективен для обычных фотографий.